# House Price Prediction — Corrected Notebook
**Fixes applied:**
- Log/exp transformation paired correctly (`log` ↔ `exp`, `log1p` ↔ `expm1`)
- `TotalSF_x_Quality` uses `ExterQualScore` consistently in both train and test
- `MedNhbdPrice` now included as a feature (it was engineered but never used)
- Single consistent `qual_map` dictionary used throughout
- XGBoost cross-validated before final submission

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error

# Load data
df = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/train.csv")
test_df = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/test.csv")

print("Train shape:", df.shape)
print("Test shape:", test_df.shape)

## Step 1 — Feature Engineering
We define a single function and apply it to both train and test.
This guarantees both datasets always get identical features — no mismatch possible.

In [ ]:
# ── Single qual_map used everywhere ──────────────────────────────────────────
# FIX: previously two dictionaries existed (qual_map and Quality_map)
# with 'Ta' vs 'TA' inconsistency. Now there is only one.
QUAL_MAP = {"Ex": 5, "Gd": 4, "TA": 3, "Fa": 2, "Po": 1}

def engineer_features(data, neigh_median=None):
    """
    Apply all feature engineering to a dataframe.
    neigh_median: pass the median computed from TRAIN only.
                  If None, compute it from data (training mode).
    """
    d = data.copy()

    # Age features
    d["HouseAge"]   = d["YrSold"] - d["YearBuilt"]
    d["RemodelAge"] = d["YrSold"] - d["YearRemodAdd"]

    # Area features
    d["TotalBath"]    = d["FullBath"] + 0.5*d["HalfBath"] + d["BsmtFullBath"] + 0.5*d["BsmtHalfBath"]
    d["TotalSF"]      = d["GrLivArea"] + d["TotalBsmtSF"]
    d["TotalPorchSF"] = d["OpenPorchSF"] + d["EnclosedPorch"] + d["3SsnPorch"] + d["ScreenPorch"]
    d["TotalGarageSF"]= d["GarageArea"].fillna(0)

    # Quality scores — using single QUAL_MAP
    d["ExterQualScore"]   = d["ExterQual"].map(QUAL_MAP)
    d["KitchenQualScore"] = d["KitchenQual"].map(QUAL_MAP).fillna(3)  # fill rare NaN with TA=3

    # Interaction features
    # FIX: TotalSF_x_Quality now uses ExterQualScore in BOTH train and test
    # Previously test used OverallQual — a different column entirely
    d["TotalSF_x_Quality"]       = d["TotalSF"]    * d["ExterQualScore"]     # FIXED
    d["Bathrooms_x_Quality"]     = d["TotalBath"]  * d["KitchenQualScore"]
    d["Bathrooms_x_OverallQual"] = d["TotalBath"]  * d["OverallQual"]

    # Neighbourhood median price
    # FIX: compute median from train only, then map to avoid data leakage
    if neigh_median is None:
        neigh_median = d.groupby("Neighborhood")["SalePrice"].median()
    d["MedNhbdPrice"] = d["Neighborhood"].map(neigh_median)

    return d, neigh_median

# Apply to train — computes neigh_median from train
df, neigh_median = engineer_features(df)

# Apply to test — passes train's neigh_median so test doesn't leak
test_df, _ = engineer_features(test_df, neigh_median=neigh_median)

print("Feature engineering complete.")
df.head(3)

## Step 2 — Select Features & Check for Nulls

In [ ]:
FEATURES = [
    "HouseAge",
    "RemodelAge",
    "TotalBath",
    "TotalSF",
    "TotalPorchSF",
    "TotalGarageSF",
    "ExterQualScore",
    "KitchenQualScore",
    "TotalSF_x_Quality",        # FIXED: consistent across train/test
    "Bathrooms_x_Quality",
    "Bathrooms_x_OverallQual",
    "MedNhbdPrice",             # FIX: was engineered before but never used!
]

print("Null counts in train features:")
print(df[FEATURES].isnull().sum())
print("\nNull counts in test features:")
print(test_df[FEATURES].isnull().sum())

In [ ]:
# Fill any remaining nulls with median (safe fallback)
for col in FEATURES:
    if df[col].isnull().any():
        df[col].fillna(df[col].median(), inplace=True)
    if test_df[col].isnull().any():
        test_df[col].fillna(df[col].median(), inplace=True)  # use TRAIN median for test

print("All nulls handled.")

## Step 3 — Target: Log1p Transformation

**FIX:** We now use `np.log1p` (log of 1+x) paired with `np.expm1` (exp(x)-1) consistently.

Previously: `np.log` was used for training but `np.expm1` was used to reverse it — these don't pair correctly and introduce small errors in every prediction.

In [ ]:
X = df[FEATURES]

# FIX: use log1p so the reverse is cleanly expm1
y = np.log1p(df["SalePrice"])

X_test_final = test_df[FEATURES]

# Check the log-transformed target looks normal
plt.figure(figsize=(8, 3))
plt.subplot(1, 2, 1)
df["SalePrice"].hist(bins=40)
plt.title("SalePrice (raw) — right skewed")
plt.subplot(1, 2, 2)
y.hist(bins=40)
plt.title("log1p(SalePrice) — more normal")
plt.tight_layout()
plt.show()

## Step 4 — Baseline: Random Forest with Cross-Validation

In [ ]:
rf_model = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)

rf_scores = cross_val_score(
    rf_model, X, y,
    scoring="neg_root_mean_squared_error",
    cv=5
)
print(f"Random Forest — 5-fold CV RMSE (log scale): {-rf_scores.mean():.4f} ± {rf_scores.std():.4f}")

# Quick train/test split to visualise predictions
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_val)

plt.figure(figsize=(6, 5))
plt.scatter(np.expm1(y_val), np.expm1(y_pred_rf), alpha=0.5)
plt.xlabel("Actual SalePrice")
plt.ylabel("Predicted SalePrice")
plt.title("Random Forest — Predicted vs Actual")
plt.plot([0, 800000], [0, 800000], 'r--', linewidth=1)  # perfect prediction line
plt.tight_layout()
plt.show()

## Step 5 — XGBoost with Cross-Validation

**FIX:** XGBoost is now cross-validated before submitting — previously it was fit and submitted without any validation.

In [ ]:
xgb_model = XGBRegressor(
    max_depth=4,
    learning_rate=0.05,
    n_estimators=1000,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    reg_alpha=0.1,      # L1 regularisation — helps prevent overfitting
    reg_lambda=1.0,     # L2 regularisation
    random_state=42,
    n_jobs=-1
)

# FIX: cross-validate XGBoost before submitting
xgb_scores = cross_val_score(
    xgb_model, X, y,
    scoring="neg_root_mean_squared_error",
    cv=5
)
print(f"XGBoost — 5-fold CV RMSE (log scale): {-xgb_scores.mean():.4f} ± {xgb_scores.std():.4f}")
print()
print("Comparison:")
print(f"  Random Forest CV RMSE: {-rf_scores.mean():.4f}")
print(f"  XGBoost CV RMSE:       {-xgb_scores.mean():.4f}")
print(f"  Winner: {'XGBoost' if -xgb_scores.mean() < -rf_scores.mean() else 'Random Forest'}")

## Step 6 — Feature Importance

In [ ]:
# Fit on full training data to get feature importances
xgb_model.fit(X, y)

importances = pd.Series(xgb_model.feature_importances_, index=FEATURES).sort_values(ascending=True)

plt.figure(figsize=(8, 5))
importances.plot(kind="barh", color="steelblue")
plt.title("XGBoost Feature Importances")
plt.xlabel("Importance score")
plt.tight_layout()
plt.show()

## Step 7 — Generate Submission

**FIX:** We trained with `log1p`, so we reverse with `expm1`. These are a correct pair.

In [ ]:
# Predict on test set
log_predictions = xgb_model.predict(X_test_final)

# FIX: use expm1 to reverse log1p — correct pairing
predictions = np.expm1(log_predictions)

# Sanity check — prices should be in a realistic range
print(f"Predicted price range: ${predictions.min():,.0f} — ${predictions.max():,.0f}")
print(f"Predicted price mean:  ${predictions.mean():,.0f}")
print(f"Actual train mean:     ${df['SalePrice'].mean():,.0f}")
print()

# If means are similar, the log/exp pairing is working correctly
submission = pd.DataFrame({
    "Id": test_df["Id"],
    "SalePrice": predictions
})

submission.to_csv("submission_xgb_corrected.csv", index=False)
print("Submission saved!")
submission.head(10)

## Summary of All Fixes

| # | Error | Fix Applied |
|---|---|---|
| 1 | `np.log` training but `np.expm1` reversal — wrong pair | Changed to `np.log1p` + `np.expm1` — correct pair |
| 2 | `TotalSF_x_Quality` used `OverallQual` in test but `ExterQualScore` in train | Now uses `ExterQualScore` in both train and test |
| 3 | `MedNhbdPrice` engineered but never used as a feature | Added to FEATURES list |
| 4 | Two conflicting qual maps (`qual_map` vs `Quality_map` with `'Ta'` typo) | Single `QUAL_MAP` used everywhere |
| 5 | XGBoost submitted without any cross-validation | 5-fold CV added before submission |
| 6 | Feature engineering applied separately to train and test (risk of mismatch) | Single `engineer_features()` function applied to both |